In [ ]:
import pandas as pd

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install-q kagglehub

ERROR: unknown command "install-q" - maybe you meant "install"


In [ ]:
import kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rabieelkharoua/alzheimers-disease-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/alzheimers-disease-dataset


In [ ]:
import os
csv_path=os.path.join(path,"alzheimers_disease_data.csv")
df=pd.read_csv(csv_path)
df.head(10)

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid
5,4756,86,1,1,1,30.626886,0,4.140144,0.211062,1.584922,...,0,0,9.015686,1,0,0,0,0,0,XXXConfid
6,4757,68,0,3,2,38.387622,1,0.646047,9.257695,5.897388,...,0,0,9.236328,0,0,0,0,1,0,XXXConfid
7,4758,75,0,0,1,18.776009,0,13.723826,4.649451,8.341903,...,0,0,4.517248,1,0,0,0,1,1,XXXConfid
8,4759,72,1,1,0,27.833188,0,12.167848,1.531360,6.736882,...,0,1,0.756232,0,0,1,0,0,0,XXXConfid
9,4760,87,0,0,0,35.456302,1,16.028688,6.440773,8.086019,...,0,1,4.554394,0,0,0,0,0,0,XXXConfid


In [ ]:
df.drop(['DoctorInCharge', 'PatientID'], axis=1, inplace=True)

In [ ]:
import joblib
import pickle
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# 1) Features & target
selected_features = [
    'MemoryComplaints',
    'BehavioralProblems',
    'ADL',
    'MMSE',
    'FunctionalAssessment'
]

X = df[selected_features].copy()
y = df['Diagnosis'].values

# 2) Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) Apply SMOTE on scaled train set
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# ------------------- Train and save each model -------------------

# 4) Train Random Forest and its scaler
scaler_rf = MinMaxScaler()
X_train_scaled_rf = scaler_rf.fit_transform(X_train_res)
X_test_scaled_rf = scaler_rf.transform(X_test)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled_rf, y_train_res)
rf_pred = rf.predict(X_test_scaled_rf)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"✅ Random Forest Accuracy: {rf_accuracy:.2f}")
# Save Random Forest model and scaler
with open("random_forest_model.pkl", "wb") as f:
    pickle.dump(rf, f)
with open("scaler_rf.pkl", "wb") as f:
    pickle.dump(scaler_rf, f)

# 5) Train Logistic Regression and its scaler
scaler_lr = MinMaxScaler()
X_train_scaled_lr = scaler_lr.fit_transform(X_train_res)
X_test_scaled_lr = scaler_lr.transform(X_test)

lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled_lr, y_train_res)
lr_pred = lr.predict(X_test_scaled_lr)
lr_accuracy = accuracy_score(y_test, lr_pred)
print(f"✅ Logistic Regression Accuracy: {lr_accuracy:.2f}")
# Save Logistic Regression model and scaler
with open("logistic_regression_model.pkl", "wb") as f:
    pickle.dump(lr, f)
with open("scaler_lr.pkl", "wb") as f:
    pickle.dump(scaler_lr, f)

# 6) Train SVM and its scaler
scaler_svm = MinMaxScaler()
X_train_scaled_svm = scaler_svm.fit_transform(X_train_res)
X_test_scaled_svm = scaler_svm.transform(X_test)

svm = SVC(random_state=42, probability=True)
svm.fit(X_train_scaled_svm, y_train_res)
svm_pred = svm.predict(X_test_scaled_svm)
svm_accuracy = accuracy_score(y_test, svm_pred)
print(f"✅ SVM Accuracy: {svm_accuracy:.2f}")
# Save SVM model and scaler
with open("svm_model.pkl", "wb") as f:
    pickle.dump(svm, f)
with open("scaler_svm.pkl", "wb") as f:
    pickle.dump(scaler_svm, f)

# 7) Train KNN and its scaler
scaler_knn = MinMaxScaler()
X_train_scaled_knn = scaler_knn.fit_transform(X_train_res)
X_test_scaled_knn = scaler_knn.transform(X_test)

knn = KNeighborsClassifier()
knn.fit(X_train_scaled_knn, y_train_res)
knn_pred = knn.predict(X_test_scaled_knn)
knn_accuracy = accuracy_score(y_test, knn_pred)
print(f"✅ KNN Accuracy: {knn_accuracy:.2f}")
# Save KNN model and scaler
with open("knn_model.pkl", "wb") as f:
    pickle.dump(knn, f)
with open("scaler_knn.pkl", "wb") as f:
    pickle.dump(scaler_knn, f)

# 8) Train Decision Tree and its scaler
scaler_dt = MinMaxScaler()
X_train_scaled_dt = scaler_dt.fit_transform(X_train_res)
X_test_scaled_dt = scaler_dt.transform(X_test)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled_dt, y_train_res)
dt_pred = dt.predict(X_test_scaled_dt)
dt_accuracy = accuracy_score(y_test, dt_pred)
print(f"✅ Decision Tree Accuracy: {dt_accuracy:.2f}")
# Save Decision Tree model and scaler
with open("decision_tree_model.pkl", "wb") as f:
    pickle.dump(dt, f)
with open("scaler_dt.pkl", "wb") as f:
    pickle.dump(scaler_dt, f)

# ------------------- Save feature list (order important!) -------------------
feature_columns = selected_features
with open("feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

print("✅ Files saved: random_forest_model.pkl, logistic_regression_model.pkl, svm_model.pkl, knn_model.pkl, decision_tree_model.pkl, scaler_rf.pkl, scaler_lr.pkl, scaler_svm.pkl, scaler_knn.pkl, scaler_dt.pkl, feature_columns.json")


✅ Random Forest Accuracy: 0.94
✅ Logistic Regression Accuracy: 0.81
✅ SVM Accuracy: 0.89
✅ KNN Accuracy: 0.88
✅ Decision Tree Accuracy: 0.87
✅ Files saved: random_forest_model.pkl, logistic_regression_model.pkl, svm_model.pkl, knn_model.pkl, decision_tree_model.pkl, scaler_rf.pkl, scaler_lr.pkl, scaler_svm.pkl, scaler_knn.pkl, scaler_dt.pkl, feature_columns.json
